# VD-EXP-002 - BDD100K YOLO11n Condition-Aware General Vehicle Detector

Bu notebook, Anomali Road Safety AI için BDD100K üzerinden 4 sınıflı araç detection fine-tune hattını kurar.

Ana hedef: `vehicle_detector_general` modelini eğitmek. Bu model yalnız normal/gündüz modeli değildir; BDD100K condition metadata korunarak `day_clear`, `night_low_light`, `rain`, `fog_low_visibility` gibi kırılımlarda ölçülür.

Ham BDD100K verisi ve model ağırlıkları Git'e eklenmez. Drive üzerinde tutulur.

## 0. Drive Yapısı

BDD100K'i resmi kaynaktan indirip Drive altında aşağıdaki gibi konumlandırın. İsterseniz `scripts/colab/download_bdd100k.py` helper'ı ile Kaggle/direct/gdown modlarından biriyle otomatik indirme yapabilirsiniz.

```text
/content/drive/MyDrive/anomali-road-safety-ai/
  datasets/
    bdd100k/
      images/100k/train/
      images/100k/val/
      labels/...
    bdd100k_vehicle_yolo/
  runs/
  exports/
```

BDD100K label dosya yolu indirilen sürüme göre değişebilir. Notebook yaygın adayları otomatik arar.

Otomatik indirme kullanılacaksa repo clone edildikten sonra örnek:

```bash
python scripts/colab/download_bdd100k.py --method kaggle --kaggle-dataset OWNER/DATASET_SLUG --extract
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics pyyaml pandas pillow tqdm

from pathlib import Path
import json
import yaml
import shutil
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO

print('Colab environment ready')

## 1. Config

`BDD_ROOT` ham BDD100K klasörüdür. `OUT_ROOT` YOLO formatına çevrilmiş güvenli çalışma klasörüdür.

In [ ]:
PROJECT_ROOT = Path('/content/drive/MyDrive/anomali-road-safety-ai')
BDD_ROOT = PROJECT_ROOT / 'datasets' / 'bdd100k'
OUT_ROOT = PROJECT_ROOT / 'datasets' / 'bdd100k_vehicle_yolo'
RUN_ROOT = PROJECT_ROOT / 'runs' / 'VD-EXP-002-yolo11n-bdd100k'

IMG_SIZE = 640
EPOCHS = 30
BATCH = 16
WORKERS = 2
SEED = 42

TARGET_CLASSES = {
    'car': 0,
    'bus': 1,
    'truck': 2,
    'motor': 3,
    'motorcycle': 3,
    'motorbike': 3,
}
NAMES = ['car', 'bus', 'truck', 'motorcycle']

for p in [OUT_ROOT, RUN_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('BDD_ROOT:', BDD_ROOT)
print('OUT_ROOT:', OUT_ROOT)
print('RUN_ROOT:', RUN_ROOT)

## 2. BDD100K Label Dosyalarını Bul

BDD100K sürümleri farklı label path kullanabilir. Aşağıdaki fonksiyon yaygın path adaylarını kontrol eder.

In [ ]:
def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError('No candidate label path found:\n' + '\n'.join(str(p) for p in paths))

TRAIN_LABEL = first_existing([
    BDD_ROOT / 'labels' / 'det_20' / 'det_train.json',
    BDD_ROOT / 'labels' / 'bdd100k_labels_images_train.json',
    BDD_ROOT / 'bdd100k_labels_images_train.json',
])
VAL_LABEL = first_existing([
    BDD_ROOT / 'labels' / 'det_20' / 'det_val.json',
    BDD_ROOT / 'labels' / 'bdd100k_labels_images_val.json',
    BDD_ROOT / 'bdd100k_labels_images_val.json',
])

TRAIN_IMG_DIR = first_existing([
    BDD_ROOT / 'images' / '100k' / 'train',
    BDD_ROOT / 'images' / 'train',
])
VAL_IMG_DIR = first_existing([
    BDD_ROOT / 'images' / '100k' / 'val',
    BDD_ROOT / 'images' / 'val',
])

print('TRAIN_LABEL:', TRAIN_LABEL)
print('VAL_LABEL:', VAL_LABEL)
print('TRAIN_IMG_DIR:', TRAIN_IMG_DIR)
print('VAL_IMG_DIR:', VAL_IMG_DIR)

## 3. Condition Mapping

Condition metadata araç sınıfı değildir. Sadece split, validation breakdown ve ileride specialist kararı için korunur.

In [ ]:
def normalize_text(value):
    if value is None:
        return 'undefined'
    return str(value).strip().lower()

def condition_from_attributes(attrs):
    attrs = attrs or {}
    weather = normalize_text(attrs.get('weather'))
    timeofday = normalize_text(attrs.get('timeofday'))
    scene = normalize_text(attrs.get('scene'))

    if weather in {'foggy', 'fog'}:
        primary = 'fog_low_visibility'
    elif weather in {'rainy', 'rain'}:
        primary = 'rain'
    elif timeofday in {'night'}:
        primary = 'night_low_light'
    elif timeofday in {'dawn/dusk', 'dawn', 'dusk'}:
        primary = 'low_light_transition'
    elif weather in {'snowy', 'snow', 'sandstorm'}:
        primary = 'adverse_other'
    elif weather in {'clear', 'overcast', 'partly cloudy'} or timeofday in {'daytime', 'day'}:
        primary = 'day_clear'
    else:
        primary = 'unknown'

    return {
        'condition_profile': primary,
        'weather': weather,
        'timeofday': timeofday,
        'scene': scene,
    }

print(condition_from_attributes({'weather': 'rainy', 'timeofday': 'night', 'scene': 'city street'}))

## 4. BDD JSON -> YOLO Label Conversion

YOLO label format:

```text
class_id x_center y_center width height
```

Koordinatlar 0-1 aralığında normalize edilir.

In [ ]:
def box2d_to_yolo(box, img_w, img_h):
    x1 = max(0.0, float(box['x1']))
    y1 = max(0.0, float(box['y1']))
    x2 = min(float(img_w), float(box['x2']))
    y2 = min(float(img_h), float(box['y2']))
    if x2 <= x1 or y2 <= y1:
        return None
    xc = ((x1 + x2) / 2.0) / img_w
    yc = ((y1 + y2) / 2.0) / img_h
    bw = (x2 - x1) / img_w
    bh = (y2 - y1) / img_h
    return xc, yc, bw, bh

def convert_split(split_name, label_json, img_dir):
    out_img_dir = OUT_ROOT / 'images' / split_name
    out_lbl_dir = OUT_ROOT / 'labels' / split_name
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    records = json.loads(Path(label_json).read_text())
    meta_rows = []
    image_list = []
    kept_boxes = 0
    skipped_images = 0

    for item in tqdm(records, desc=f'convert {split_name}'):
        name = item.get('name')
        src_img = img_dir / name
        if not src_img.exists():
            skipped_images += 1
            continue

        with Image.open(src_img) as im:
            img_w, img_h = im.size

        lines = []
        for obj in item.get('labels', []) or []:
            category = normalize_text(obj.get('category'))
            if category not in TARGET_CLASSES:
                continue
            box = obj.get('box2d')
            if not box:
                continue
            converted = box2d_to_yolo(box, img_w, img_h)
            if converted is None:
                continue
            cls_id = TARGET_CLASSES[category]
            xc, yc, bw, bh = converted
            lines.append(f'{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')

        if not lines:
            continue

        dst_img = out_img_dir / name
        if not dst_img.exists():
            try:
                dst_img.symlink_to(src_img)
            except Exception:
                shutil.copy2(src_img, dst_img)

        (out_lbl_dir / f'{Path(name).stem}.txt').write_text('\n'.join(lines) + '\n')
        cond = condition_from_attributes(item.get('attributes'))
        image_path = str(dst_img)
        image_list.append(image_path)
        meta_rows.append({
            'split': split_name,
            'image': image_path,
            'name': name,
            'vehicle_box_count': len(lines),
            **cond,
        })
        kept_boxes += len(lines)

    return meta_rows, image_list, kept_boxes, skipped_images

train_meta, train_images, train_boxes, train_skipped = convert_split('train', TRAIN_LABEL, TRAIN_IMG_DIR)
val_meta, val_images, val_boxes, val_skipped = convert_split('val', VAL_LABEL, VAL_IMG_DIR)

metadata = pd.DataFrame(train_meta + val_meta)
(OUT_ROOT / 'metadata').mkdir(exist_ok=True, parents=True)
metadata.to_csv(OUT_ROOT / 'metadata' / 'condition_metadata.csv', index=False)

print('train images:', len(train_images), 'train boxes:', train_boxes, 'skipped:', train_skipped)
print('val images:', len(val_images), 'val boxes:', val_boxes, 'skipped:', val_skipped)
print(metadata.groupby(['split', 'condition_profile']).size())

## 5. YOLO `data.yaml` ve Condition Split Listeleri

In [ ]:
splits_dir = OUT_ROOT / 'splits'
splits_dir.mkdir(parents=True, exist_ok=True)

(splits_dir / 'train.txt').write_text('\n'.join(train_images) + '\n')
(splits_dir / 'val.txt').write_text('\n'.join(val_images) + '\n')

data_yaml = {
    'path': str(OUT_ROOT),
    'train': str(splits_dir / 'train.txt'),
    'val': str(splits_dir / 'val.txt'),
    'names': {i: name for i, name in enumerate(NAMES)},
}
(OUT_ROOT / 'data.yaml').write_text(yaml.safe_dump(data_yaml, sort_keys=False))

condition_yamls = {}
val_df = metadata[metadata['split'] == 'val'].copy()
for condition, group in val_df.groupby('condition_profile'):
    condition_list = splits_dir / f'val_{condition}.txt'
    condition_list.write_text('\n'.join(group['image'].tolist()) + '\n')
    condition_yaml = OUT_ROOT / f'data_val_{condition}.yaml'
    condition_yaml.write_text(yaml.safe_dump({
        'path': str(OUT_ROOT),
        'train': str(splits_dir / 'train.txt'),
        'val': str(condition_list),
        'names': {i: name for i, name in enumerate(NAMES)},
    }, sort_keys=False))
    condition_yamls[condition] = condition_yaml

print('data.yaml:', OUT_ROOT / 'data.yaml')
print('condition yamls:', condition_yamls)

## 6. Train YOLO11n

İlk gerçek fine-tune deneyi `VD-EXP-002` olarak kaydedilir.

In [ ]:
model = YOLO('yolo11n.pt')

train_result = model.train(
    data=str(OUT_ROOT / 'data.yaml'),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    workers=WORKERS,
    seed=SEED,
    project=str(RUN_ROOT),
    name='train',
    exist_ok=True,
)

print(train_result)

## 7. Overall Validation

In [ ]:
best_weights = RUN_ROOT / 'train' / 'weights' / 'best.pt'
trained = YOLO(str(best_weights))
overall_metrics = trained.val(data=str(OUT_ROOT / 'data.yaml'), imgsz=IMG_SIZE, project=str(RUN_ROOT), name='val_overall', exist_ok=True)
print(overall_metrics)

## 8. Condition Breakdown Validation

Bu bölüm aynı modeli condition-specific validation listelerinde ölçer. Specialist kararı bu breakdown sonuçlarına göre verilir.

In [ ]:
rows = []
for condition, condition_yaml in condition_yamls.items():
    print('Validating condition:', condition)
    metrics = trained.val(data=str(condition_yaml), imgsz=IMG_SIZE, project=str(RUN_ROOT), name=f'val_{condition}', exist_ok=True)
    rows.append({
        'experiment_id': 'VD-EXP-002',
        'model': 'YOLO11n',
        'condition_profile': condition,
        'val_image_count': int((metadata['split'].eq('val') & metadata['condition_profile'].eq(condition)).sum()),
        'map50': float(metrics.box.map50),
        'map5095': float(metrics.box.map),
        'precision_mean': float(metrics.box.mp),
        'recall_mean': float(metrics.box.mr),
    })

breakdown = pd.DataFrame(rows).sort_values('condition_profile')
breakdown_path = RUN_ROOT / 'condition_breakdown_metrics.csv'
breakdown.to_csv(breakdown_path, index=False)
breakdown

## 9. Export

Export çıktıları Drive'da tutulur, Git'e eklenmez.

In [ ]:
EXPORT_DIR = PROJECT_ROOT / 'exports' / 'VD-EXP-002-yolo11n-bdd100k'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

onnx_path = trained.export(format='onnx', imgsz=IMG_SIZE, dynamic=True)
print('ONNX export:', onnx_path)

shutil.copy2(best_weights, EXPORT_DIR / 'best.pt')
if Path(onnx_path).exists():
    shutil.copy2(onnx_path, EXPORT_DIR / Path(onnx_path).name)

print('Export dir:', EXPORT_DIR)

## 10. Sonraki Adım

Bu notebook tamamlandıktan sonra:

1. Overall ve condition breakdown metriklerini repo içindeki benchmark CSV'ye özet olarak işleyin.
2. `best.pt` dosyasını MacBook local edge runtime benchmark için repo dışı güvenli konuma alın.
3. `night_low_light`, `rain`, `fog_low_visibility` kırılımlarında general model zayıfsa specialist kararı için not açın.
4. Specialist model eğitmeden önce `best_general` kararını netleştirin.